# Phase 1 A3 Distillation Smoke Plan and Manual Hold

Notebook marker: `a3_distillation_smoke_v1_20260420`

Purpose:

- Prepare the next Phase 1 comparator implementation step after A2/A2b, A2c and A2d non-claim smoke reviews.
- Verify that the repo contains the CLI-backed A3 distillation smoke runner, config, tests and model-card revision note.
- Hash-link the locked prereg identity and reviewed upstream smoke sources before any launch.
- Default to manual hold. The first run after pulling this notebook should keep `RUN_A3_DISTILLATION_SMOKE = False`.

Scientific boundary:

- This notebook is orchestration only. Durable implementation logic must live in `src/` and be covered by tests.
- A3 smoke uses an internal training-scalp-feature teacher proxy to validate distillation mechanics and leakage guards.
- The smoke teacher is not a final iEEG teacher and not evidence of privileged transfer.
- Any BA/ECE/Brier values produced later are implementation diagnostics only.
- Do not claim decoder efficacy, A3 distillation efficacy, A3 superiority/inferiority, privileged-transfer efficacy, or full Phase 1 performance from this notebook.

## Technical basis checked before launch

The V5.5 technical documents describe A3 as the blind/full teacher-distillation comparator that protects interpretation of the A4 privileged-transfer claim. The headline claim remains closed unless A4 beats A3 and the other strong scalp-only comparators under the locked calibration, influence and control rules.

For this repo revision, notebook 12 does **not** implement final A3. It launches only a bounded A3 implementation smoke:

- scalp feature extraction is delegated to `src/phase1/a3_smoke.py`;
- normalization is fit on training subjects only;
- the smoke teacher proxy is fit on training subjects only;
- teacher outputs used for student fitting are generated only for training rows;
- student fitting uses training subjects only;
- inference uses the student probe on scalp features only, with no teacher outputs at test time;
- all artifacts are marked non-claim.

This notebook must not introduce alternate distillation logic. It only calls `python -m src.cli phase1_real --a3-smoke` after the manual hold is intentionally released.

In [ ]:
# Cell 1 - Runtime, clone/pull, optional install, and unit tests.
# First pass expectation: keep INSTALL_SIGNAL_EXTRAS = False and RUN_A3_DISTILLATION_SMOKE = False later.
# If you later launch real EDF extraction, set INSTALL_SIGNAL_EXTRAS = True and rerun this cell plus tests.

from __future__ import annotations

import base64
import getpass
import hashlib
import importlib.util
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

NOTEBOOK_FIX_MARKER = 'a3_distillation_smoke_v1_20260420'
INSTALL_SIGNAL_EXTRAS = False
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')


def _redacted_cmd(cmd):
    redacted = []
    skip_next = False
    for item in cmd:
        if skip_next:
            redacted.append('<redacted>')
            skip_next = False
            continue
        text = str(item)
        if text == '-c':
            redacted.append(text)
            skip_next = True
        elif 'Authorization:' in text or 'github_pat_' in text:
            redacted.append('<redacted>')
        else:
            redacted.append(text)
    return ' '.join(redacted)


def run(cmd, cwd=None, check=True, capture=False, env=None):
    print('$', _redacted_cmd([str(x) for x in cmd]))
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE if capture else None,
        stderr=subprocess.PIPE if capture else None,
        env=env,
    )
    if capture:
        if completed.stdout:
            print(completed.stdout)
        if completed.stderr:
            print(completed.stderr, file=sys.stderr)
    if check and completed.returncode:
        raise subprocess.CalledProcessError(completed.returncode, cmd, output=completed.stdout, stderr=completed.stderr)
    return completed


if not REPO_DIR.exists():
    token = getpass.getpass('Nhap GitHub token cho repo private: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    header = f'Authorization: Basic {auth}'
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo already exists:', REPO_DIR)
    run(['git', 'fetch', 'origin'], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

env = os.environ.copy()
if INSTALL_SIGNAL_EXTRAS:
    env['INSTALL_SIGNAL_EXTRAS'] = '1'
run(['bash', 'bootstrap/install_runtime.sh'], cwd=REPO_DIR, env=env)
unit_result = run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=False)
if unit_result.returncode != 0:
    print('Unit tests failed. Do not launch A3 smoke until the repo test suite is clean.')
    raise subprocess.CalledProcessError(unit_result.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])
print('Unit tests passed. Continue to source-chain checks.')

In [ ]:
# Cell 2 - Source-of-truth paths.
# These are explicit reviewed paths; avoid silently following latest.txt for claim-affecting provenance.

DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
DATASET_ROOT = DRIVE_ROOT / 'data' / 'ds004752'
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

PREREG_BUNDLE = ARTIFACT_ROOT / 'prereg' / '20260418T161442014597Z' / 'prereg_bundle.json'
EXPECTED_LOCKED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PHASE1_READINESS_RUN = ARTIFACT_ROOT / 'phase1_readiness' / '20260419T154005857077Z'

A2_A2B_REVIEWED_RUN = ARTIFACT_ROOT / 'phase1_model_smoke' / '20260419T172746816598Z'
A2C_REVIEWED_RUN = ARTIFACT_ROOT / 'phase1_a2c_smoke' / '20260420T081425018260Z'
A2D_REVIEWED_RUN = ARTIFACT_ROOT / 'phase1_a2d_smoke' / '20260420T074006419555Z'
GAP_REVIEW_RUN = ARTIFACT_ROOT / 'phase1_gap_review' / '20260420T085031775625Z'

A3_PLAN_ROOT = ARTIFACT_ROOT / 'phase1_a3_smoke_plan'
A3_OUTPUT_ROOT = ARTIFACT_ROOT / 'phase1_a3_smoke'

for label, path in {
    'PREREG_BUNDLE': PREREG_BUNDLE,
    'PHASE1_READINESS_RUN': PHASE1_READINESS_RUN,
    'A2_A2B_REVIEWED_RUN': A2_A2B_REVIEWED_RUN,
    'A2C_REVIEWED_RUN': A2C_REVIEWED_RUN,
    'A2D_REVIEWED_RUN': A2D_REVIEWED_RUN,
    'GAP_REVIEW_RUN': GAP_REVIEW_RUN,
    'DATASET_ROOT': DATASET_ROOT,
}.items():
    print(label, 'exists =', path.exists(), '|', path)

In [ ]:
# Cell 3 - Small audit helpers used only by this notebook.

def utc_stamp() -> str:
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')


def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json(path: Path, data) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding='utf-8')


def latest_run(root: Path) -> Path | None:
    latest = root / 'latest.txt'
    if latest.exists():
        target = Path(latest.read_text(encoding='utf-8').strip())
        if target.exists():
            return target
    if not root.exists():
        return None
    runs = sorted([p for p in root.iterdir() if p.is_dir()])
    return runs[-1] if runs else None

In [ ]:
# Cell 4 - Validate locked prereg identity and reviewed upstream smoke/gap notes.
# This cell compares the identity hash stored inside the bundle, not the raw file SHA256.

bundle = read_json(PREREG_BUNDLE)
locked_identity_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_file_sha256 = sha256_file(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', locked_identity_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert locked_identity_hash == EXPECTED_LOCKED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash does not match reviewed hash.'

review_notes = {
    'A2/A2b review note': (A2_A2B_REVIEWED_RUN / 'phase1_a2_a2b_model_smoke_review_note.json', 'phase1_a2_a2b_model_smoke_review_pass_non_claim'),
    'A2c review note': (A2C_REVIEWED_RUN / 'phase1_a2c_coral_smoke_review_note.json', 'phase1_a2c_coral_smoke_review_pass_non_claim'),
    'A2d review note': (A2D_REVIEWED_RUN / 'phase1_a2d_riemannian_smoke_review_note.json', 'phase1_a2d_riemannian_smoke_review_pass_non_claim'),
}
for label, (path, expected_status) in review_notes.items():
    print(label, 'exists =', path.exists(), '|', path)
    assert path.exists(), f'Missing reviewed source: {path}'
    observed = read_json(path).get('status')
    print(label, 'status =', observed)
    assert observed == expected_status, f'{label} status mismatch: {observed}'

gap_summary = GAP_REVIEW_RUN / 'phase1_comparator_suite_gap_review_summary.json'
print('Gap review summary exists =', gap_summary.exists(), '|', gap_summary)
assert gap_summary.exists(), 'Missing gap review output. Run notebook 11 before A3 planning.'
gap = read_json(gap_summary)
print('Gap review claim_ready:', gap.get('claim_ready'))
print('Gap review next_step:', gap.get('next_step'))
assert gap.get('claim_ready') is False, 'Gap review must keep claim_ready false before A3 smoke.'

print('Source-chain review passed. Prior smoke outputs remain non-claim diagnostics only.')

In [ ]:
# Cell 5 - Hash relevant code/config/doc surfaces for the A3 plan.

tracked_sources = [
    REPO_DIR / 'src' / 'phase1' / 'a3_smoke.py',
    REPO_DIR / 'src' / 'cli.py',
    REPO_DIR / 'configs' / 'phase1' / 'a3_smoke.json',
    REPO_DIR / 'configs' / 'models' / 'distill_a3.yaml',
    REPO_DIR / 'tests' / 'unit' / 'test_phase1_a3_smoke.py',
    REPO_DIR / 'docs' / '01_v55_doc_code_crosswalk.md',
    REPO_DIR / 'docs' / '02_colab_local_runbook.md',
    REPO_DIR / 'notebooks' / '12_colab_phase1_a3_distillation_smoke.ipynb',
]
source_hashes = {}
for path in tracked_sources:
    source_hashes[str(path.relative_to(REPO_DIR))] = {
        'exists': path.exists(),
        'sha256': sha256_file(path) if path.exists() else None,
    }
print(json.dumps(source_hashes, indent=2))
assert all(item['exists'] for item in source_hashes.values()), 'A3 source/config/notebook/test surface is incomplete.'

In [ ]:
# Cell 6 - A3 runner preflight.
# mne/signal extras are required only when launching real EDF extraction, not for this manual-hold planning pass.

preflight = {
    'notebook_fix_marker': NOTEBOOK_FIX_MARKER,
    'runner_module_exists': (REPO_DIR / 'src' / 'phase1' / 'a3_smoke.py').exists(),
    'phase_config_exists': (REPO_DIR / 'configs' / 'phase1' / 'a3_smoke.json').exists(),
    'distill_config_exists': (REPO_DIR / 'configs' / 'models' / 'distill_a3.yaml').exists(),
    'numpy_available': importlib.util.find_spec('numpy') is not None,
    'mne_available': importlib.util.find_spec('mne') is not None,
}
cli_help = run(['python', '-m', 'src.cli', 'phase1_real', '--help'], cwd=REPO_DIR, check=False, capture=True)
preflight['cli_help_returncode'] = cli_help.returncode
preflight['cli_exposes_a3_smoke'] = '--a3-smoke' in (cli_help.stdout or '')
import_check = run(['python', '-c', 'from src.phase1.a3_smoke import run_phase1_a3_smoke; print("a3 import ok")'], cwd=REPO_DIR, check=False, capture=True)
preflight['runner_import_returncode'] = import_check.returncode
preflight['runner_import_available'] = import_check.returncode == 0
distill_text = (REPO_DIR / 'configs' / 'models' / 'distill_a3.yaml').read_text(encoding='utf-8') if preflight['distill_config_exists'] else ''
preflight['distill_revision_note_present'] = 'smoke_revision_non_claim' in distill_text
preflight['distill_not_placeholder'] = 'placeholder_until_prereg' not in distill_text

blockers = []
if not preflight['runner_module_exists']:
    blockers.append('src/phase1/a3_smoke.py is missing')
if not preflight['phase_config_exists']:
    blockers.append('configs/phase1/a3_smoke.json is missing')
if not preflight['cli_exposes_a3_smoke']:
    blockers.append('CLI does not expose --a3-smoke')
if not preflight['runner_import_available']:
    blockers.append('A3 runner import failed')
if not preflight['numpy_available']:
    blockers.append('numpy is required for the A3 smoke proxy')
if not preflight['distill_revision_note_present'] or not preflight['distill_not_placeholder']:
    blockers.append('configs/models/distill_a3.yaml does not contain the non-claim smoke revision note')
preflight['blockers'] = blockers
print(json.dumps(preflight, indent=2))

## A3 smoke artifact contract

If the manual hold is later released, the CLI run must write and review these artifacts under `artifacts/phase1_a3_smoke/<timestamp>/`:

- `phase1_a3_smoke_inputs.json`
- `phase1_a3_smoke_summary.json`
- `phase1_a3_smoke_report.md`
- `a3_metrics_smoke.json`
- `a3_distillation_audit.json`
- `a3_teacher_manifest.json`
- `a3_feature_manifest.json`
- `calibration_a3_smoke_report.json`
- `negative_controls_a3_smoke_report.json`
- `influence_a3_smoke_report.json`
- `fold_logs/*.json`
- `a3_logits_smoke/*.json`

Required interpretation guard:

- `claim_ready` must be false.
- `final_a3_comparator` must be false.
- `does_not_estimate_privileged_transfer_efficacy` must be true.
- The teacher manifest must state `real_ieeg_teacher_used=false` and `privileged_features_used=false` for this smoke proxy.
- Outer-test subjects must not appear in normalization fit, teacher fit, teacher outputs for student fit, student fit, or calibration fit.
- Teacher outputs must not be used at inference.

In [ ]:
# Cell 7 - Write the A3 plan artifact.

plan_stamp = utc_stamp()
plan_dir = A3_PLAN_ROOT / plan_stamp
plan = {
    'status': 'phase1_a3_distillation_smoke_plan_recorded',
    'created_utc': plan_stamp,
    'notebook_fix_marker': NOTEBOOK_FIX_MARKER,
    'locked_prereg_identity_hash': locked_identity_hash,
    'raw_prereg_file_sha256': raw_prereg_file_sha256,
    'prereg_bundle': str(PREREG_BUNDLE),
    'phase1_readiness_run': str(PHASE1_READINESS_RUN),
    'dataset_root': str(DATASET_ROOT),
    'a2_a2b_reviewed_run': str(A2_A2B_REVIEWED_RUN),
    'a2c_reviewed_run': str(A2C_REVIEWED_RUN),
    'a2d_reviewed_run': str(A2D_REVIEWED_RUN),
    'gap_review_run': str(GAP_REVIEW_RUN),
    'a3_output_root': str(A3_OUTPUT_ROOT),
    'preflight': preflight,
    'source_hashes': source_hashes,
    'planned_cli': [
        'python', '-m', 'src.cli', 'phase1_real',
        '--profile', 't4_safe',
        '--config', str(PREREG_BUNDLE),
        '--readiness-run', str(PHASE1_READINESS_RUN),
        '--dataset-root', str(DATASET_ROOT),
        '--output-root', str(A3_OUTPUT_ROOT),
        '--a3-smoke',
        '--phase-config', 'configs/phase1/a3_smoke.json',
        '--max-outer-folds', '2',
    ],
    'non_claim_debug_only': True,
    'claim_ready': False,
    'launch_policy': 'manual_hold_default_false; explicit acknowledgement required before training',
    'not_ok_to_claim': [
        'decoder_efficacy',
        'A3_distillation_efficacy',
        'A3_final_comparator_performance',
        'A3_superiority_or_inferiority_against_other_comparators',
        'privileged_transfer_efficacy',
        'full_phase1_neural_comparator_performance',
    ],
}
write_json(plan_dir / 'phase1_a3_distillation_smoke_plan.json', plan)
(A3_PLAN_ROOT / 'latest.txt').write_text(str(plan_dir), encoding='utf-8')
print(json.dumps({
    'status': plan['status'],
    'plan_dir': str(plan_dir),
    'runner_available': not bool(preflight['blockers']),
    'blockers': preflight['blockers'],
    'locked_prereg_identity_hash': locked_identity_hash,
}, indent=2))

In [ ]:
# Cell 8 - Manual launch gate.
# First rerun after pull should keep RUN_A3_DISTILLATION_SMOKE = False.
# To launch later, set INSTALL_SIGNAL_EXTRAS = True in Cell 1, rerun bootstrap/tests, then set the flag and ACK below.

RUN_A3_DISTILLATION_SMOKE = False
MANUAL_LAUNCH_ACK = ''
REQUIRED_ACK = 'I reviewed the A3 distillation plan and understand this is non-claim implementation smoke'

launch_record = {
    'created_utc': utc_stamp(),
    'run_flag': RUN_A3_DISTILLATION_SMOKE,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'preflight_blockers': preflight['blockers'],
    'mne_available': preflight['mne_available'],
    'plan_dir': str(plan_dir),
}

if preflight['blockers']:
    launch_record['status'] = 'phase1_a3_distillation_smoke_blocked_preflight'
    write_json(plan_dir / 'phase1_a3_distillation_smoke_launch_record.json', launch_record)
    print(json.dumps(launch_record, indent=2))
    raise RuntimeError('A3 runner/config/CLI preflight has blockers. Do not launch.')
elif not RUN_A3_DISTILLATION_SMOKE:
    launch_record['status'] = 'phase1_a3_distillation_smoke_manual_hold'
    write_json(plan_dir / 'phase1_a3_distillation_smoke_launch_record.json', launch_record)
    print(json.dumps(launch_record, indent=2))
    print('HELD: Runner appears available, but training was not launched because manual flag is False.')
    print('NEXT: review this plan artifact, then only later rerun with signal extras and explicit acknowledgement if appropriate.')
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not launch A3 smoke without explicit non-claim acknowledgement.')
elif not preflight['mne_available']:
    raise RuntimeError('A3 real EDF feature extraction requires mne/signal extras. Set INSTALL_SIGNAL_EXTRAS = True in Cell 1, rerun bootstrap/tests, then relaunch intentionally.')
else:
    launch_record['status'] = 'phase1_a3_distillation_smoke_launch_approved'
    write_json(plan_dir / 'phase1_a3_distillation_smoke_launch_record.json', launch_record)
    cmd = plan['planned_cli']
    run(cmd, cwd=REPO_DIR)
    print('A3 distillation smoke command completed. Review generated artifacts before any further run.')

In [ ]:
# Cell 9 - Closeout summary for the manual-hold pass or later launch pass.

latest = latest_run(A3_OUTPUT_ROOT)
print('================ Phase 1 A3 Distillation Smoke Closeout ================')
print('Notebook fix marker:', NOTEBOOK_FIX_MARKER)
print('Plan source of truth:', plan_dir)
print('Runner available:', not bool(preflight['blockers']))
print('Run flag:', RUN_A3_DISTILLATION_SMOKE)
print('Expected locked prereg identity hash:', EXPECTED_LOCKED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', locked_identity_hash)
print('Raw prereg file SHA256:', raw_prereg_file_sha256)
print('Blockers:', preflight['blockers'])

if latest and RUN_A3_DISTILLATION_SMOKE:
    print('Latest A3 output:', latest)
    expected = [
        'phase1_a3_smoke_inputs.json',
        'phase1_a3_smoke_summary.json',
        'phase1_a3_smoke_report.md',
        'a3_metrics_smoke.json',
        'a3_distillation_audit.json',
        'a3_teacher_manifest.json',
        'a3_feature_manifest.json',
        'calibration_a3_smoke_report.json',
        'negative_controls_a3_smoke_report.json',
        'influence_a3_smoke_report.json',
    ]
    for name in expected:
        print(name, 'exists =', (latest / name).exists())
    summary_path = latest / 'phase1_a3_smoke_summary.json'
    if summary_path.exists():
        summary = read_json(summary_path)
        print(json.dumps({
            'status': summary.get('status'),
            'comparator': summary.get('comparator'),
            'n_outer_folds': summary.get('n_outer_folds'),
            'blockers': summary.get('blockers'),
            'claim_ready': summary.get('claim_ready'),
            'final_a3_comparator': summary.get('final_a3_comparator'),
            'scientific_limit': summary.get('scientific_limit'),
        }, indent=2))
    print('CHECK REQUIRED: A3 smoke command was launched. Review fold logs, distillation audit, teacher manifest and metrics before continuing.')
elif preflight['blockers']:
    print('BLOCKED: A3 runner/config/CLI is unavailable. Fix repo implementation first.')
else:
    print('HELD: Runner appears available, but training was not launched because manual flag is False.')
    print('NEXT: close this first-pass notebook after confirming the plan artifact, then relaunch only with explicit non-claim acknowledgement if appropriate.')

print()
print('NOT OK TO CLAIM: This notebook does not prove decoder efficacy, A3 distillation efficacy, or privileged-transfer efficacy.')

In [ ]:
# Cell 10 - Optional post-launch review note.
# This cell writes a review note only after RUN_A3_DISTILLATION_SMOKE=True and required artifacts pass integrity checks.

if not RUN_A3_DISTILLATION_SMOKE:
    print('SKIP: A3 smoke was not launched. No review note is written during manual hold.')
elif latest is None:
    raise RuntimeError('A3 smoke was launched but no latest output directory was found.')
else:
    required = [
        'phase1_a3_smoke_inputs.json',
        'phase1_a3_smoke_summary.json',
        'phase1_a3_smoke_report.md',
        'a3_metrics_smoke.json',
        'a3_distillation_audit.json',
        'a3_teacher_manifest.json',
        'a3_feature_manifest.json',
        'calibration_a3_smoke_report.json',
        'negative_controls_a3_smoke_report.json',
        'influence_a3_smoke_report.json',
    ]
    missing = [name for name in required if not (latest / name).exists()]
    if missing:
        raise RuntimeError(f'Missing required A3 smoke artifacts: {missing}')

    summary = read_json(latest / 'phase1_a3_smoke_summary.json')
    audit = read_json(latest / 'a3_distillation_audit.json')
    teacher = read_json(latest / 'a3_teacher_manifest.json')
    metrics = read_json(latest / 'a3_metrics_smoke.json')
    feature_manifest = read_json(latest / 'a3_feature_manifest.json')

    checks = []
    if summary.get('status') == 'phase1_a3_distillation_smoke_complete':
        checks.append('a3_smoke_completed')
    if summary.get('n_outer_folds') == 2:
        checks.append('two_outer_folds_completed')
    if summary.get('comparator') == 'A3_distillation':
        checks.append('comparator_limited_to_A3_distillation')
    if summary.get('claim_ready') is False:
        checks.append('claim_ready_false')
    if summary.get('final_a3_comparator') is False:
        checks.append('final_a3_comparator_false')
    if audit.get('outer_test_subject_used_for_any_fit') is False:
        checks.append('outer_test_subject_not_used_for_any_fit')
    if teacher.get('real_ieeg_teacher_used') is False and teacher.get('privileged_features_used') is False:
        checks.append('teacher_manifest_marks_internal_scalp_proxy_only')
    if teacher.get('teacher_used_at_inference') is False:
        checks.append('teacher_not_used_at_inference')
    if all(f.get('no_outer_test_subject_in_any_fit') for f in audit.get('folds', [])):
        checks.append('fold_level_no_outer_test_subject_in_any_fit')
    if metrics.get('summary', {}).get('claim_ready') is False:
        checks.append('metrics_marked_non_claim')

    review_note = {
        'status': 'phase1_a3_distillation_smoke_review_pass_non_claim',
        'created_utc': utc_stamp(),
        'reviewed_run': str(latest),
        'scope': '2-fold A3 distillation implementation smoke only',
        'checks_passed': checks,
        'folds': [
            {
                'fold_id': fold.get('fold_id'),
                'outer_test_subject': fold.get('outer_test_subject'),
                'no_outer_test_subject_in_any_fit': fold.get('no_outer_test_subject_in_any_fit'),
                'teacher_used_at_inference': fold.get('teacher_used_at_inference'),
                'real_ieeg_teacher_used': fold.get('real_ieeg_teacher_used'),
                'privileged_features_used': fold.get('privileged_features_used'),
            }
            for fold in audit.get('folds', [])
        ],
        'metric_summary_recorded_as_diagnostic_only': metrics.get('summary'),
        'feature_manifest': {
            'n_rows': feature_manifest.get('n_rows'),
            'n_features': feature_manifest.get('n_features'),
            'subjects': feature_manifest.get('subjects'),
            'skipped_sessions_count': len(feature_manifest.get('skipped_sessions', [])),
            'read_fallbacks_count': len(feature_manifest.get('read_fallbacks', [])),
        },
        'metrics_interpretation': {
            'allowed': 'Implementation diagnostics only.',
            'not_allowed': 'Do not use BA/ECE/Brier values as Phase 1 efficacy evidence, final A3 evidence, or privileged-transfer evidence.',
        },
        'next_allowed_steps': [
            'Commit/push any notebook or runner hardening if this review found issues.',
            'Optionally run another bounded A3 smoke only if there is a specific engineering question.',
            'Proceed to A4 planning/implementation or full comparator-suite readiness only under prereg/revision discipline.',
            'Do not open headline Phase 1 claims until full comparator suite, controls, calibration, influence and reporting package are run under prereg/revision policy.',
        ],
        'not_ok_to_claim': [
            'decoder efficacy',
            'A3 distillation efficacy',
            'A3 final comparator performance',
            'A3 superiority or inferiority against A2/A2b/A2c/A2d/A4',
            'privileged-transfer efficacy',
            'full Phase 1 neural comparator performance',
        ],
    }
    expected_checks = {
        'a3_smoke_completed',
        'two_outer_folds_completed',
        'comparator_limited_to_A3_distillation',
        'claim_ready_false',
        'final_a3_comparator_false',
        'outer_test_subject_not_used_for_any_fit',
        'teacher_manifest_marks_internal_scalp_proxy_only',
        'teacher_not_used_at_inference',
        'fold_level_no_outer_test_subject_in_any_fit',
        'metrics_marked_non_claim',
    }
    missing_checks = sorted(expected_checks - set(checks))
    if missing_checks:
        raise RuntimeError(f'A3 review checks failed or missing: {missing_checks}')
    note_path = latest / 'phase1_a3_distillation_smoke_review_note.json'
    write_json(note_path, review_note)
    print('Review note written:', note_path)
    print(json.dumps(review_note, indent=2, ensure_ascii=False))